# 02 · Train & evaluate — YOLOv8 PPE detection on construction sites
**MAICEN-0526 · M4U3 Computer Vision · Group 7**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/solomon8909/maicen0526-m4u3-ppe-detection/blob/main/notebooks/02_train_eval.ipynb)

This notebook reproduces the full pipeline from a clean Colab runtime: dataset download from Roboflow → YOLOv8 fine-tuning → validation metrics → curves → evidence pack → false-positive / false-negative mining → reproducibility record.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Disconnect and delete runtime` and `Run all`. No Roboflow account, no API key and no Colab secrets are needed. Edit only the configuration cell.

| `RUN_MODE` | What happens | Typical runtime (T4) |
|---|---|---|
| `"full"` | Trains for `EPOCHS`, evaluates the new weights | ~40–75 min |
| `"verify"` | 5-epoch verification run, then evaluates the **released** weights | ~10–15 min |
| `"load"` | No training; downloads the released weights and evaluates them | ~5 min |

If no GPU is available the notebook switches `"full"` to `"verify"` automatically and says so in the output.

## 1 · Configuration (the only cell you should edit)

In [ ]:
RUN_MODE      = "full"          # "full" | "verify" | "load"
MODEL_VARIANT = "yolov8n.pt"    # yolov8n.pt (fast) or yolov8s.pt (more accurate, ~2x slower)
EPOCHS        = 30
VERIFY_EPOCHS = 5
BATCH         = 16
IMGSZ         = 640
SEED          = 42

# ---- Primary data path: frozen dataset published as a GitHub Release asset. No account, no key. ----
DATA_URL      = "https://github.com/solomon8909/maicen0526-m4u3-ppe-detection/releases/download/v1.0/ppe-construction-v1-yolo11.zip"
DATA_SHA256   = "979ebedaa790feb32be28f93d96dc034ac0f71bbdfaeb589746cc61d8b83c926"
NEW_IMAGES_URL = "https://github.com/solomon8909/maicen0526-m4u3-ppe-detection/releases/download/v1.0/new-images.zip"
NEW_IMAGES_SHA256 = "5d3597fc72a5e0d2b38a7eeaa969aa80ad6c13df59ad543d9bb93404b1958dde"
WEIGHTS_URL   = "https://github.com/solomon8909/maicen0526-m4u3-ppe-detection/releases/download/v1.0/best.pt"

# ---- Secondary path only (section 4b). Never required for a clean run. ----
RF_WORKSPACE  = "solomon-yirga"
RF_PROJECT    = "construction-site-safety-cnfob"
RF_VERSION    = 1               # pinned version number - never "latest"

ULTRALYTICS_VERSION = ""        # after the first successful run, paste the printed version here to pin it
CONF_THRESHOLD = 0.25           # confidence used for evidence images and FP/FN mining
IOU_MATCH      = 0.50           # IoU needed for a prediction to count as a correct detection

## 2 · Environment

In [ ]:
import subprocess, sys, time, datetime, platform, os
pkg = f"ultralytics=={ULTRALYTICS_VERSION}" if ULTRALYTICS_VERSION else "ultralytics"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg, "roboflow"], check=True)

import torch, ultralytics
from pathlib import Path
T_START = time.time()
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
if GPU == "CPU only" and RUN_MODE == "full":
    RUN_MODE = "verify"
    print("⚠️  No GPU detected — RUN_MODE switched from 'full' to 'verify'. This is documented in the reproducibility record.")
print(f"Python {platform.python_version()} | torch {torch.__version__} | ultralytics {ultralytics.__version__} | device: {GPU} | mode: {RUN_MODE}")

WORK = Path("/content") if Path("/content").exists() else Path.cwd()
OUT = WORK / "results"
for d in ["curves", "metrics", "evidence/annotation_examples", "evidence/validation_predictions",
          "evidence/new_image_predictions", "evidence/false_positives", "evidence/false_negatives"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)

## 3 · Helper functions
Dataset path repair, label parsing, same-class IoU matching for FP/FN mining, drawing and reporting. Collapsed by default in Colab — no need to edit.

In [ ]:
# ---- Helper functions: dataset handling, box matching, drawing, reporting ----
import os, glob, random, shutil
from pathlib import Path
import numpy as np
import cv2
import yaml

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def sha256_file(path, chunk=1 << 20):
    """Chunked hashing — a 1-2 GB zip read in one go can kill a Colab session."""
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def fetch_release_zip(url, sha256, dest_dir, zip_path):
    """Keyless download of a frozen dataset from a GitHub Release, checksum-verified."""
    import urllib.request
    dest_dir, zip_path = Path(dest_dir), Path(zip_path)
    if list(dest_dir.rglob("data.yaml")) or (dest_dir.exists() and any(dest_dir.rglob("*.jpg"))):
        print(f"Already present at {dest_dir} — skipping download.")
        return dest_dir
    dest_dir.mkdir(parents=True, exist_ok=True)
    if zip_path.exists():
        zip_path.unlink()
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, zip_path)
    digest = sha256_file(zip_path)
    if sha256 and digest.lower() != sha256.strip().lower():   # PowerShell prints uppercase, Python lowercase
        zip_path.unlink(missing_ok=True)   # delete the partial file so the next run re-downloads
        raise AssertionError(f"Checksum mismatch.\n  expected {sha256}\n  got      {digest}")
    print(f"SHA256 verified: {digest}")
    import zipfile
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest_dir)
    return dest_dir


def prepare_data_yaml(root):
    """Find data.yaml (Roboflow zips unpack into a nested <project>-<version>/ folder),
    pin `path` to the real extract root and set the split paths relative to it."""
    hits = list(Path(root).rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    cfg_path = hits[0]
    dataset_root = cfg_path.parent
    cfg = yaml.safe_load(cfg_path.read_text())
    cfg["path"] = str(dataset_root)
    cfg["train"] = "train/images"
    cfg["val"] = "valid/images"
    if (dataset_root / "test" / "images").is_dir():
        cfg["test"] = "test/images"
    else:
        cfg.pop("test", None)
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return cfg_path, cfg


def split_dir(cfg, split):
    """Absolute images folder for a split."""
    return Path(cfg["path"]) / cfg[split]


def list_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXT)


def label_path_for(img_path):
    img_path = Path(img_path)
    return img_path.parent.parent / "labels" / (img_path.stem + ".txt")


def read_yolo_labels(img_path, w, h):
    """Return (N,4) xyxy pixel boxes and (N,) class ids from a YOLO .txt label."""
    lp = label_path_for(img_path)
    boxes, cls = [], []
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            c, cx, cy, bw, bh = int(float(parts[0])), *map(float, parts[1:5])
            boxes.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
            cls.append(c)
    return np.array(boxes, dtype=float).reshape(-1, 4), np.array(cls, dtype=int)


def dataset_summary(data_cfg):
    """Images and labelled instances per class per split."""
    names = data_cfg["names"]
    names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
    rows = []
    for split in ("train", "val", "test"):
        if split not in data_cfg:
            continue
        imgs = list_images(split_dir(data_cfg, split))
        counts = np.zeros(len(names), dtype=int)
        for im in imgs:
            lp = label_path_for(im)
            if lp.exists():
                for line in lp.read_text().strip().splitlines():
                    if line.strip():
                        counts[int(float(line.split()[0]))] += 1
        rows.append({"split": split, "images": len(imgs), **{n: int(c) for n, c in zip(names, counts)}})
    return rows, names


def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def match_detections(gt_boxes, gt_cls, pr_boxes, pr_cls, pr_conf, iou_thr=0.5):
    """Greedy same-class matching, highest confidence first.
    Returns indices of false-positive predictions and false-negative ground truths."""
    order = np.argsort(-pr_conf) if len(pr_conf) else np.array([], dtype=int)
    ious = iou_matrix(pr_boxes, gt_boxes)
    gt_used = np.zeros(len(gt_boxes), dtype=bool)
    fp = []
    for i in order:
        cand = np.where((gt_cls == pr_cls[i]) & (~gt_used))[0]
        if len(cand):
            j = cand[np.argmax(ious[i, cand])]
            if ious[i, j] >= iou_thr:
                gt_used[j] = True
                continue
        fp.append(int(i))
    fn = [int(j) for j in np.where(~gt_used)[0]]
    return fp, fn


PALETTE = [(56, 56, 255), (151, 157, 255), (31, 112, 255), (29, 178, 255), (49, 210, 207),
           (10, 249, 72), (23, 204, 146), (134, 219, 61), (52, 147, 26), (187, 212, 0)]


def draw_boxes(img, boxes, cls, names, conf=None, color=None, thick=2, prefix=""):
    out = img.copy()
    for k, (b, c) in enumerate(zip(boxes, cls)):
        col = color or PALETTE[int(c) % len(PALETTE)]
        x1, y1, x2, y2 = map(int, b)
        cv2.rectangle(out, (x1, y1), (x2, y2), col, thick)
        label = f"{prefix}{names[int(c)]}" + (f" {conf[k]:.2f}" if conf is not None else "")
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        ty = y1 if y1 - th - 6 >= 0 else y1 + th + 6          # label inside the box if it would leave the image
        cv2.rectangle(out, (x1, ty - th - 6), (x1 + tw + 4, ty), col, -1)
        cv2.putText(out, label, (x1 + 2, ty - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def banner(img, text):
    bar = np.full((32, img.shape[1], 3), 30, dtype=np.uint8)
    scale = 0.6
    while scale > 0.3 and cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 1)[0][0] > img.shape[1] - 16:
        scale -= 0.05                                         # shrink long titles to fit narrow images
    cv2.putText(bar, text, (8, 22), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 1, cv2.LINE_AA)
    return np.vstack([bar, img])


def side_by_side(left, right, left_title="Ground truth", right_title="Prediction"):
    h = max(left.shape[0], right.shape[0])
    pad = lambda im: np.vstack([im, np.zeros((h - im.shape[0], im.shape[1], 3), dtype=np.uint8)])
    return np.hstack([banner(pad(left), left_title), np.full((h + 32, 6, 3), 255, np.uint8), banner(pad(right), right_title)])


def md_table(rows, cols, fmt=None):
    fmt = fmt or {}
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for r in rows:
        cells = []
        for c in cols:
            v = r.get(c, "")
            cells.append(fmt[c].format(v) if c in fmt and isinstance(v, (int, float, np.floating)) else str(v))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def plot_training_curves(results_csv, out_png):
    import pandas as pd
    import matplotlib.pyplot as plt
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    ep = df["epoch"]
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for c in ("train/box_loss", "val/box_loss", "train/cls_loss", "val/cls_loss"):
        if c in df: ax[0].plot(ep, df[c], label=c)
    ax[0].set_title("Losses"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=8)
    for c in ("metrics/precision(B)", "metrics/recall(B)"):
        if c in df: ax[1].plot(ep, df[c], label=c.split("/")[1])
    ax[1].set_title("Precision / Recall (val)"); ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1); ax[1].legend(fontsize=8)
    for c in ("metrics/mAP50(B)", "metrics/mAP50-95(B)"):
        if c in df: ax[2].plot(ep, df[c], label=c.split("/")[1])
    ax[2].set_title("mAP (val)"); ax[2].set_xlabel("epoch"); ax[2].set_ylim(0, 1); ax[2].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(out_png, dpi=150); plt.close(fig)
    return out_png

## 4 · Dataset — keyless download from the GitHub Release

The dataset this project's results depend on is a **frozen zip published as a Release asset** on this repository. It downloads over plain HTTPS with no account and no API key, and its SHA256 checksum is verified before use, so the file you get is provably the file the model was trained on.

Roboflow is where the dataset was annotated and versioned; it is not on the reproduction path. A secondary Roboflow cell is kept in section 4b and does not run when the keyless dataset is already present.

In [ ]:
DATA_ROOT = WORK / "dataset"
fetch_release_zip(DATA_URL, DATA_SHA256, DATA_ROOT, WORK / "dataset.zip")
DATA_YAML, data_cfg = prepare_data_yaml(DATA_ROOT)
print(data_cfg)

### 4b · Secondary path — Roboflow (optional, skipped automatically)
Runs only if the keyless dataset is missing. It does not crash when no secret is set, and it pins the exact version number.

In [ ]:
if not list(DATA_ROOT.rglob("data.yaml")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "roboflow"], check=True)
    try:
        from google.colab import userdata
        api_key = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        from getpass import getpass
        api_key = getpass("Roboflow API key (any account works for public Universe datasets): ")
    from roboflow import Roboflow
    rf = Roboflow(api_key=api_key)
    rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download(
        "yolov11", location=str(WORK / "dataset_roboflow"))
    DATA_YAML, data_cfg = prepare_data_yaml(WORK / "dataset_roboflow")
else:
    print("Keyless dataset already present; skipping Roboflow download.")

### 4c · Dataset summary
Image and instance counts per split — paste this into the README's Dataset section.

In [ ]:
rows, NAMES = dataset_summary(data_cfg)
summary_md = md_table(rows, ["split", "images"] + NAMES)
(OUT / "metrics" / "dataset_summary.md").write_text(summary_md)
print(summary_md)
tot = sum(r["images"] for r in rows)
print("\nSplit: " + " / ".join(f"{r['split']} {100*r['images']/tot:.0f}%" for r in rows))

## 5 · Training

In [ ]:
from ultralytics import YOLO
TRAIN_DIR, TRAIN_MIN, EPOCHS_RUN = None, 0.0, 0
if RUN_MODE in ("full", "verify"):
    EPOCHS_RUN = EPOCHS if RUN_MODE == "full" else VERIFY_EPOCHS
    t0 = time.time()
    model = YOLO(MODEL_VARIANT)
    model.train(data=str(DATA_YAML), epochs=EPOCHS_RUN, imgsz=IMGSZ, batch=BATCH, seed=SEED,
                deterministic=True, project=str(WORK / "runs"), name=f"train_{RUN_MODE}", exist_ok=True, plots=True)
    TRAIN_DIR = Path(model.trainer.save_dir)
    TRAIN_MIN = (time.time() - t0) / 60
    print(f"Training finished in {TRAIN_MIN:.1f} min → {TRAIN_DIR}")
else:
    print("RUN_MODE = 'load' → training skipped.")

## 6 · Select weights to evaluate
`full` evaluates the weights just trained. `verify` and `load` evaluate the released weights, so the reported metrics always come from the real 30-epoch model.

In [ ]:
import urllib.request
if RUN_MODE == "full":
    EVAL_WEIGHTS = TRAIN_DIR / "weights" / "best.pt"
else:
    EVAL_WEIGHTS = WORK / "weights" / "best.pt"
    EVAL_WEIGHTS.parent.mkdir(exist_ok=True)
    if not EVAL_WEIGHTS.exists():
        urllib.request.urlretrieve(WEIGHTS_URL, EVAL_WEIGHTS)
print("Evaluating:", EVAL_WEIGHTS, f"({EVAL_WEIGHTS.stat().st_size/1e6:.1f} MB)")
model = YOLO(str(EVAL_WEIGHTS))

## 7 · Validation metrics (Precision · Recall · mAP50 · mAP50–95)

In [ ]:
m = model.val(data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH, split="val", plots=True,
              project=str(WORK / "runs"), name="eval", exist_ok=True)
EVAL_DIR = Path(m.save_dir)
overall = {"class": "all", "precision": float(m.box.mp), "recall": float(m.box.mr),
           "mAP50": float(m.box.map50), "mAP50-95": float(m.box.map)}
per_class = [{"class": m.names[int(c)], "precision": float(m.box.p[i]), "recall": float(m.box.r[i]),
              "mAP50": float(m.box.ap50[i]), "mAP50-95": float(m.box.ap[i])}
             for i, c in enumerate(m.box.ap_class_index)]
cols = ["class", "precision", "recall", "mAP50", "mAP50-95"]
fmt = {c: "{:.3f}" for c in cols[1:]}
METRICS_MD = md_table([overall] + per_class, cols, fmt)
(OUT / "metrics" / "metrics_table.md").write_text(METRICS_MD)
import csv
with open(OUT / "metrics" / "metrics_table.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows([overall] + per_class)
print(METRICS_MD)

## 8 · Curves and plots

In [ ]:
from IPython.display import Image, display
copied = []
for src_dir in [d for d in (TRAIN_DIR, EVAL_DIR) if d is not None]:
    for p in list(src_dir.glob("*.png")) + list(src_dir.glob("results.csv")):
        if p.name.startswith(("train_batch", "val_batch")):
            continue
        dst = OUT / "curves" / (("train_" if src_dir == TRAIN_DIR else "eval_") + p.name)
        shutil.copy(p, dst); copied.append(dst.name)
if TRAIN_DIR is not None and (TRAIN_DIR / "results.csv").exists():
    plot_training_curves(TRAIN_DIR / "results.csv", OUT / "curves" / "training_curves_summary.png")
    display(Image(str(OUT / "curves" / "training_curves_summary.png"), width=1000))
print("Saved:", sorted(copied))
for name in ("eval_confusion_matrix_normalized.png", "eval_BoxPR_curve.png", "eval_PR_curve.png"):
    if (OUT / "curves" / name).exists():
        display(Image(str(OUT / "curves" / name), width=650))

## 9 · Evidence pack
Annotation examples (ground-truth labels), 10 validation predictions shown against ground truth, and predictions on the unseen images downloaded from the Release.

In [ ]:
random.seed(SEED)
train_imgs = list_images(split_dir(data_cfg, "train")); val_imgs = list_images(split_dir(data_cfg, "val"))

# 9a · 5 annotation examples
for k, p in enumerate(random.sample(train_imgs, min(5, len(train_imgs))), 1):
    img = cv2.imread(str(p)); b, c = read_yolo_labels(p, img.shape[1], img.shape[0])
    cv2.imwrite(str(OUT / "evidence/annotation_examples" / f"annotation_{k:02d}.jpg"),
                banner(draw_boxes(img, b, c, NAMES), f"Annotation example {k} - {len(b)} labels"))

# 9b · 10 validation predictions (ground truth | prediction)
for k, p in enumerate(random.sample(val_imgs, min(10, len(val_imgs))), 1):
    img = cv2.imread(str(p)); gb, gc = read_yolo_labels(p, img.shape[1], img.shape[0])
    r = model.predict(str(p), conf=CONF_THRESHOLD, imgsz=IMGSZ, verbose=False)[0]
    pb, pc, pf = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()
    cv2.imwrite(str(OUT / "evidence/validation_predictions" / f"val_pred_{k:02d}.jpg"),
                side_by_side(draw_boxes(img, gb, gc, NAMES), draw_boxes(img, pb, pc, NAMES, pf)))

# 9c · new-image predictions
NEW_DIR = WORK / "new_images"
fetch_release_zip(NEW_IMAGES_URL, NEW_IMAGES_SHA256, NEW_DIR, WORK / "new_images.zip")
new_imgs = list_images(NEW_DIR) or [p for d in NEW_DIR.rglob("*") if d.is_dir() for p in list_images(d)]
if not new_imgs:
    print("⚠️  No images found in the new-images release asset.")
for k, p in enumerate(new_imgs, 1):
    r = model.predict(str(p), conf=CONF_THRESHOLD, imgsz=IMGSZ, verbose=False)[0]
    img = cv2.imread(str(p))
    pb, pc, pf = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()
    cv2.imwrite(str(OUT / "evidence/new_image_predictions" / f"new_pred_{k:02d}_{p.stem}.jpg"),
                banner(draw_boxes(img, pb, pc, NAMES, pf), f"New image {k}: {p.name} - {len(pb)} detections @conf {CONF_THRESHOLD}"))

for sub in ("annotation_examples", "validation_predictions", "new_image_predictions"):
    print(f"{sub}: {len(list((OUT/'evidence'/sub).glob('*.jpg')))} images")
display(Image(str(sorted((OUT / "evidence/validation_predictions").glob("*.jpg"))[0]), width=1000))

## 10 · Error mining — false positives and false negatives
Every validation image is matched prediction-to-label (same class, IoU ≥ `IOU_MATCH`). Unmatched predictions are **false positives**; unmatched labels are **false negatives**. The most confident FPs and the largest (most obvious) FNs are saved as images for `docs/error_analysis.md`. Green = label, red = prediction.

In [ ]:
fp_rows, fn_rows = [], []
fp_count = {n: 0 for n in NAMES}; fn_count = {n: 0 for n in NAMES}
for p in val_imgs:
    img = cv2.imread(str(p)); gb, gc = read_yolo_labels(p, img.shape[1], img.shape[0])
    r = model.predict(str(p), conf=CONF_THRESHOLD, imgsz=IMGSZ, verbose=False)[0]
    pb, pc, pf = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()
    fp, fn = match_detections(gb, gc, pb, pc, pf, IOU_MATCH)
    for i in fp:
        fp_count[NAMES[pc[i]]] += 1
        fp_rows.append({"image": p.name, "class": NAMES[pc[i]], "conf": float(pf[i]), "path": p, "box": pb[i], "cls": pc[i], "gb": gb, "gc": gc})
    for j in fn:
        fn_count[NAMES[gc[j]]] += 1
        area = (gb[j][2]-gb[j][0])*(gb[j][3]-gb[j][1]) / (img.shape[0]*img.shape[1])
        fn_rows.append({"image": p.name, "class": NAMES[gc[j]], "area_pct": 100*area, "path": p, "box": gb[j], "cls": gc[j], "pb": pb, "pc": pc, "pf": pf})

def save_case(row, kind, k):
    img = cv2.imread(str(row["path"]))
    if kind == "fp":
        base = draw_boxes(img, row["gb"], row["gc"], NAMES, color=(60, 180, 60), prefix="GT ")
        out = draw_boxes(base, [row["box"]], [row["cls"]], NAMES, [row["conf"]], color=(40, 40, 230), thick=3, prefix="FP ")
        title = f"FP {k}: predicted {row['class']} ({row['conf']:.2f}) with no matching label - {row['image']}"
    else:
        base = draw_boxes(img, row["pb"], row["pc"], NAMES, row["pf"], color=(40, 40, 230), prefix="pred ")
        out = draw_boxes(base, [row["box"]], [row["cls"]], NAMES, color=(60, 180, 60), thick=3, prefix="MISSED ")
        title = f"FN {k}: labelled {row['class']} not detected - {row['image']}"
    cv2.imwrite(str(OUT / f"evidence/{'false_positives' if kind=='fp' else 'false_negatives'}" / f"{kind}_{k:02d}.jpg"), banner(out, title))

top_fp = sorted(fp_rows, key=lambda r: -r["conf"])[:6]
top_fn = sorted(fn_rows, key=lambda r: -r["area_pct"])[:6]
for k, r in enumerate(top_fp, 1): save_case(r, "fp", k)
for k, r in enumerate(top_fn, 1): save_case(r, "fn", k)

err_md = md_table([{"class": n, "false_positives": fp_count[n], "false_negatives": fn_count[n]} for n in NAMES],
                  ["class", "false_positives", "false_negatives"])
cases = "\n".join([f"- FP {k}: `{r['image']}` — predicted **{r['class']}** at {r['conf']:.2f}" for k, r in enumerate(top_fp, 1)] +
                   [f"- FN {k}: `{r['image']}` — missed **{r['class']}** ({r['area_pct']:.1f}% of image)" for k, r in enumerate(top_fn, 1)])
(OUT / "metrics" / "error_counts.md").write_text(err_md + "\n\n" + cases + "\n")
print(f"Validation images: {len(val_imgs)} | FP total: {len(fp_rows)} | FN total: {len(fn_rows)} (conf ≥ {CONF_THRESHOLD}, IoU ≥ {IOU_MATCH})\n")
print(err_md); print(); print(cases)

## 11 · Reproducibility record
Written to `results/reproducibility_record.md`. Paste the printed block into the README's *Reproducibility proof* section.

In [ ]:
total_min = (time.time() - T_START) / 60
data_file = DATA_URL.rsplit("/", 1)[-1]
data_path_note = "keyless GitHub Release (no credentials)" if str(DATA_ROOT) in str(DATA_YAML) else "Roboflow secondary path"
split_str = " / ".join(f"{r['split']} {r['images']}" for r in rows)
mode_note = " (verification run; metrics from released weights)" if RUN_MODE != "full" else ""
epochs_str = str(EPOCHS_RUN) if RUN_MODE != "load" else "-"
run_utc = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
record = f"""### Reproducibility proof
- **Last successful run:** {run_utc}
- **Run mode:** `{RUN_MODE}`{mode_note}
- **Hardware:** {GPU}
- **Software:** Python {platform.python_version()} · torch {torch.__version__} · ultralytics {ultralytics.__version__}
- **Dataset:** frozen release asset `{data_file}` · SHA256 `{DATA_SHA256}` · {split_str} images
- **Data path used:** {data_path_note} · annotated and versioned in Roboflow `{RF_WORKSPACE}/{RF_PROJECT}` v{RF_VERSION}
- **Model / parameters:** {MODEL_VARIANT} · epochs {epochs_str} (released model: {EPOCHS}) · batch {BATCH} · imgsz {IMGSZ} · seed {SEED}
- **Training time:** {TRAIN_MIN:.1f} min · **Total notebook runtime:** {total_min:.1f} min
- **Expected runtime:** full ≈ 40–75 min on T4 · verify ≈ 10–15 min · load ≈ 5 min

**Validation metrics**

{METRICS_MD}
"""
(OUT / "reproducibility_record.md").write_text(record)
print(record)

## 12 · Download outputs
Downloads `results.zip` (upload its contents to the repo's `/results` folder) and, after a `full` run, `best.pt` (attach to the GitHub Release `v1.0`).

In [ ]:
shutil.make_archive(str(WORK / "results"), "zip", OUT)
try:
    from google.colab import files
    files.download(str(WORK / "results.zip"))
    if RUN_MODE == "full":
        files.download(str(EVAL_WEIGHTS))
except Exception as e:
    print("Not in Colab — outputs are at:", WORK / "results.zip", EVAL_WEIGHTS)